# Robustness Checks Before Step 3

Two checks planned in `docs/pre_development_research_report.md` (Sections 6 and 7) before the Step 2 configuration becomes the new baseline:

1. **Re-check on fresh folds (selection bias).** Step 2 chose the best of 14 variants on the same 15 folds (CV seed 0), so its score is slightly optimistic. Here the Step 1 baseline, the best single Step 2 change (`word_1_2`) and the selected Step 2 configuration are compared again on folds from an unused seed (100). The selection is confirmed if the Step 2 configuration again passes the decision rule (positive mean paired PR-AUC gain and better in at least 12 of 15 folds).
2. **Label noise.** The similarity groups that contain both real and fake postings are inspected, and the Step 2 configuration is re-evaluated with these postings removed from every training fold (sensitivity analysis). Labels are not changed, and the postings are still evaluated when they fall in a test fold, so both runs are scored on exactly the same rows.

The test set is not used in this notebook.

In [1]:
import sklearn

if sklearn.__version__ != "1.6.1":
    raise RuntimeError(
        f"This notebook uses scikit-learn 1.6.1 (found {sklearn.__version__}). "
        "Install the pinned versions with: pip install -r requirements.txt"
    )

import time
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

from pathlib import Path

import numpy as np
import pandas as pd

from fakejob.data import load_split
from fakejob.evaluation import compare, passes_rule, repeated_group_cv
from fakejob.features import STEP2_CONFIG, Config, build_pipeline, posting_features
from fakejob.models import TrainingRowFilter

pd.set_option("display.precision", 3)
RESULTS_DIR = Path("results")
METRICS = ["average_precision", "roc_auc", "f1", "precision", "recall"]

split = load_split()
features = posting_features(split.model_df)
dev_index = split.dev_index
X_dev = features.loc[dev_index]
y_dev = split.y.loc[dev_index]
groups_dev = split.groups.loc[dev_index]
print(f"Development data: {len(dev_index)} postings, {int(y_dev.sum())} fake")

Development data: 15001 postings, 724 fake


## Helper

`run()` evaluates an estimator with the Step 1 protocol (5 × 3 repeated `StratifiedGroupKFold`, nested threshold) for a given CV seed. `judge()` applies the decision rule, as in Step 2.

In [2]:
def run(name, estimator, seed, n_jobs):
    start = time.time()
    result = repeated_group_cv(
        estimator, X_dev, y_dev, groups_dev, name=name,
        n_splits=5, n_repeats=3, inner_splits=3, seed=seed, n_jobs=n_jobs,
    )
    result.folds.to_csv(RESULTS_DIR / f"checks_{name}_folds.csv", index=False)
    print(f"{name}: {time.time() - start:.0f}s")
    return result

def judge(reference, candidates):
    rows = []
    for cand in candidates:
        comparison = compare(reference, cand, metrics=METRICS)
        ap, f1 = comparison.loc["average_precision"], comparison.loc["f1"]
        rows.append({
            "variant": cand.name,
            "PR-AUC": f"{cand.folds['average_precision'].mean():.3f} ± {cand.folds['average_precision'].std():.3f}",
            "Δ PR-AUC": ap["mean_diff"],
            "PR-AUC wins": ap["folds_improved"],
            "F1": f"{cand.folds['f1'].mean():.3f} ± {cand.folds['f1'].std():.3f}",
            "Δ F1": f1["mean_diff"],
            "F1 wins": f1["folds_improved"],
            "passes rule": passes_rule(comparison),
        })
    return pd.DataFrame(rows).set_index("variant")

def summary_table(*results):
    return pd.concat([r.summary() for r in results], axis=1).loc[["threshold"] + METRICS]

## Check 1: Fresh Folds

Seed 100 gives new outer splits for all three repeats (seeds 100–102) and new inner splits; none of them was used in Step 1 or Step 2.

In [3]:
FRESH_SEED = 100
word_1_2_config = Config(word_ngrams=(1, 2), min_df=2)

fresh_baseline = run("fresh_baseline", build_pipeline(Config()), FRESH_SEED, n_jobs=-1)
fresh_word_1_2 = run("fresh_word_1_2", build_pipeline(word_1_2_config), FRESH_SEED, n_jobs=-1)
fresh_step2 = run("fresh_step2", build_pipeline(STEP2_CONFIG), FRESH_SEED, n_jobs=4)

display(summary_table(fresh_baseline, fresh_word_1_2, fresh_step2))
print("Against the baseline:")
display(judge(fresh_baseline, [fresh_word_1_2, fresh_step2]))
print("Step 2 configuration against the best single change:")
display(judge(fresh_word_1_2, [fresh_step2]))

fresh_baseline: 34s


fresh_word_1_2: 71s


fresh_step2: 681s


,fresh_baseline,fresh_word_1_2,fresh_step2
threshold,-0.048 ± 0.067,-0.212 ± 0.044,-0.254 ± 0.046
average_precision,0.825 ± 0.050,0.868 ± 0.042,0.876 ± 0.040
roc_auc,0.976 ± 0.008,0.984 ± 0.005,0.984 ± 0.006
f1,0.747 ± 0.057,0.786 ± 0.045,0.801 ± 0.053
precision,0.790 ± 0.061,0.832 ± 0.064,0.851 ± 0.064
recall,0.714 ± 0.083,0.750 ± 0.069,0.762 ± 0.074


Against the baseline:


,PR-AUC,Δ PR-AUC,PR-AUC wins,F1,Δ F1,F1 wins,passes rule
variant,,,,,,,
fresh_word_1_2,0.868 ± 0.042,0.043,15/15,0.786 ± 0.045,0.039,15/15,True
fresh_step2,0.876 ± 0.040,0.051,15/15,0.801 ± 0.053,0.054,15/15,True


Step 2 configuration against the best single change:


,PR-AUC,Δ PR-AUC,PR-AUC wins,F1,Δ F1,F1 wins,passes rule
variant,,,,,,,
fresh_step2,0.876 ± 0.040,0.008,13/15,0.801 ± 0.053,0.015,12/15,True


### Same comparison on the original folds (seed 0)

Results saved in Step 2, shown next to the fresh-fold results. If the gain on fresh folds is clearly smaller than on the original folds, part of the Step 2 gain was a selection effect.

In [4]:
def gain(base_folds, cand_folds, metric="average_precision"):
    return (cand_folds[metric] - base_folds[metric]).mean()

original = {name: pd.read_csv(RESULTS_DIR / f"step2_{name}_folds.csv")
            for name in ["baseline", "word_1_2", "combined"]}
pd.DataFrame({
    "original folds (seed 0)": {
        "baseline PR-AUC": original["baseline"]["average_precision"].mean(),
        "Step 2 PR-AUC": original["combined"]["average_precision"].mean(),
        "Δ PR-AUC vs baseline": gain(original["baseline"], original["combined"]),
        "Δ PR-AUC vs word_1_2": gain(original["word_1_2"], original["combined"]),
        "Δ F1 vs baseline": gain(original["baseline"], original["combined"], "f1"),
    },
    "fresh folds (seed 100)": {
        "baseline PR-AUC": fresh_baseline.folds["average_precision"].mean(),
        "Step 2 PR-AUC": fresh_step2.folds["average_precision"].mean(),
        "Δ PR-AUC vs baseline": gain(fresh_baseline.folds, fresh_step2.folds),
        "Δ PR-AUC vs word_1_2": gain(fresh_word_1_2.folds, fresh_step2.folds),
        "Δ F1 vs baseline": gain(fresh_baseline.folds, fresh_step2.folds, "f1"),
    },
})

,original folds (seed 0),fresh folds (seed 100)
baseline PR-AUC,0.825,0.825
Step 2 PR-AUC,0.871,0.876
Δ PR-AUC vs baseline,0.045,0.051
Δ PR-AUC vs word_1_2,0.010,0.008
Δ F1 vs baseline,0.041,0.054


## Check 2: Label Noise

A similarity group is *mixed* when it contains both real and fake postings. Groups are built from description similarity, so a mixed group means that near-identical descriptions carry different labels. The table lists every mixed group in the whole dataset with the part of the split it belongs to; only development groups are used below.

In [5]:
data = split.model_df.assign(group=split.groups, fraudulent=split.y)
part = pd.Series("test", index=data.index)
part[split.train_index] = "train"
part[split.val_index] = "val"
data["part"] = part

group_stats = data.groupby("group")["fraudulent"].agg(["size", "sum"])
mixed_groups = group_stats.index[(group_stats["sum"] > 0) & (group_stats["sum"] < group_stats["size"])]
mixed = data[data["group"].isin(mixed_groups)]

overview = mixed.groupby("group").agg(
    part=("part", "first"), postings=("fraudulent", "size"), fake=("fraudulent", "sum"),
)
print(f"Groups with more than one posting: {(group_stats['size'] > 1).sum()}, mixed: {len(mixed_groups)}")
overview

Groups with more than one posting: 1767, mixed: 8

,part,postings,fake
group,,,
389,val,19,1
1375,test,7,3
2283,train,18,2
2287,train,9,1
2691,train,5,3
5143,train,2,1
6946,train,6,1
11711,train,7,1


### What do the fake postings in mixed groups look like?

For each mixed group: the fake postings next to the real ones, with the fields that differ most often (company profile, logo, location, salary).

In [6]:
def describe(row):
    profile = row["company_profile"]
    return pd.Series({
        "group": row["group"],
        "fraudulent": row["fraudulent"],
        "title": str(row["title"])[:45],
        "location": row["location"],
        "company_profile": "<empty>" if pd.isna(profile) or not str(profile).strip() else str(profile)[:40] + "…",
        "has_company_logo": row["has_company_logo"],
        "salary_range": row["salary_range"],
    })

examples = (
    mixed.sort_values(["group", "fraudulent"], ascending=[True, False])
    .groupby("group").head(4)
    .apply(describe, axis=1)
    .set_index(["group", "fraudulent"])
)
examples

title  \
group fraudulent                                                  
389   1                                     Final Expense Agent   
      0                       Engagement Manager (South Africa)   
      0                                        Product Manager    
      0                                   Connectivity Engineer   
1375  1                                              Data entry   
      1               Administrative Assistant/Customer Service   
      1                                              Data Entry   
      0                                              Data Entry   
2283  1                           Bilingual Retail Demonstrator   
      1                                     Retail Demonstrator   
      0                                            Demonstrator   
      0                                            Demonstrator   
2287  1                  Nursing & Therapy Positions- Home Care   
      0                                    Associate Consultant   
      0                         Standards & Practices Executive   
      0                             Business Development Expert   
2691  1           Immediate Opening : Oracle Developer for  Bah   
      1           Urgently Required System Administrator for  B   
      1           Immediate Opening : Linux Administrator for B   
      0              Urgent Need :Oracle Developer for Bahrain.   
5143  1                             Brand & Logo Design Contest   
      0                          Brand & Logo Design Contest!!!   
6946  1                           Front-End/JavaScript Engineer   
      0                                         Rails Developer   
      0                                  Java Software Engineer   
      0                              Developer (Front-end team)   
11711 1           PeopleSoft Financial Consultant Required for    
      0           Mobile Application Developer Requiredfor  UAE   
      0                   Immediate Opening : Dotnet Developer    
      0                   Sharepoint Developer Required for UAE   

                               location  \
group fraudulent                          
389   1                          US, ,    
      0                          ZA, ,    
      0                    GR, , Athens   
      0                   GR, I, Athens   
1375  1              US, TX, Fort Worth   
      1               US, CA, San diego   
      1           US, FL, HILLIARD FL     
      0                   US, FL, miami   
2283  1               US, CA, San Diego   
      1                 US, WA, Seattle   
      0                 US, WA, Seattle   
      0                  US, CA, Dublin   
2287  1                          US, ,    
      0                          GR, ,    
      0                  IN, MH, Mumbai   
      0                  IN, GJ, 380013   
2691  1                          BH, ,    
      1                        BH, 13,    
      1                              BH   
      0                          BH, ,    
5143  1            US, MI, Grand Rapids   
      0            US, MI, Grand Rapids   
6946  1                 ID, JW, Jakarta   
      0                   GR, I, Athens   
      0                   GR, I, Athens   
      0                   GR, I, Athens   
11711 1                          AE, ,    
      0                   AE, DU, Dubai   
      0                   AE, DU, Dubai   
      0                  AE, , Abudhabi   

                                            company_profile  has_company_logo  \
group fraudulent                                                                
389   1                                             <empty>                 0   
      0           Upstream’s mission is to revolutionise t…                 1   
      0           Upstream’s mission is to revolutionise t…                 1   
      0           Upstream’s mission is to revolutionise t…                 1   
1375  1                                           

In [7]:
comparison = mixed.groupby("fraudulent").agg(
    postings=("title", "size"),
    company_profile_empty=("company_profile", lambda s: s.isna().mean()),
    has_company_logo=("has_company_logo", "mean"),
)
comparison.index = comparison.index.map({0: "real", 1: "fake"})
comparison

,postings,company_profile_empty,has_company_logo
fraudulent,,,
real,60,0.15,0.85
fake,13,1.00,0.00


### Sensitivity analysis

The Step 2 configuration is re-run on the original folds (seed 0) with the development postings of mixed groups removed from every training fold. The reference is the Step 2 result saved on the same folds. Both runs are evaluated on exactly the same test-fold rows. This is not an adoption test: the question is whether the results change in a meaningful way, so the mean paired difference is compared with the fold-to-fold variation.

In [8]:
X_dev_flagged = X_dev.assign(in_mixed_group=groups_dev.isin(mixed_groups).astype(int))
n_flagged = int(X_dev_flagged["in_mixed_group"].sum())
n_flagged_fake = int(y_dev[X_dev_flagged["in_mixed_group"] == 1].sum())
print(f"Development postings in mixed groups: {n_flagged} ({n_flagged_fake} fake)")

start = time.time()
without_mixed = repeated_group_cv(
    TrainingRowFilter(build_pipeline(STEP2_CONFIG), "in_mixed_group"),
    X_dev_flagged, y_dev, groups_dev, name="step2_without_mixed",
    n_splits=5, n_repeats=3, inner_splits=3, seed=0, n_jobs=4,
)
without_mixed.folds.to_csv(RESULTS_DIR / "checks_step2_without_mixed_folds.csv", index=False)
print(f"step2_without_mixed: {time.time() - start:.0f}s")

class Saved:
    def __init__(self, name, folds):
        self.name, self.folds = name, folds

    def summary(self):
        stats = self.folds[["threshold"] + METRICS].agg(["mean", "std"]).T
        return pd.Series({m: f"{r['mean']:.3f} ± {r['std']:.3f}" for m, r in stats.iterrows()}, name=self.name)

step2_original = Saved("step2 (all training rows)", original["combined"])
display(summary_table(step2_original, without_mixed))
compare(step2_original, without_mixed, metrics=METRICS)

Development postings in mixed groups: 66 (10 fake)


step2_without_mixed: 1263s


,step2 (all training rows),step2_without_mixed
threshold,-0.272 ± 0.070,-0.294 ± 0.057
average_precision,0.871 ± 0.047,0.864 ± 0.051
roc_auc,0.984 ± 0.007,0.983 ± 0.008
f1,0.789 ± 0.057,0.791 ± 0.058
precision,0.834 ± 0.065,0.843 ± 0.054
recall,0.752 ± 0.066,0.748 ± 0.072


,baseline,candidate,mean_diff,std_diff,folds_improved
metric,,,,,
average_precision,0.871,0.864,-6.566e-03,0.013,5/15
roc_auc,0.984,0.983,-9.362e-04,0.002,6/15
f1,0.789,0.791,2.011e-03,0.018,9/15
precision,0.834,0.843,9.678e-03,0.043,7/15
recall,0.752,0.748,-4.241e-03,0.022,6/15


## Findings

**1. The Step 2 selection holds on fresh folds.** On folds from an unused seed (100), the Step 2 configuration again passes the decision rule against the baseline (PR-AUC 0.825 → 0.876, better in 15/15 folds; F1 0.747 → 0.801, 15/15) and against the best single change `word_1_2` (+0.008 PR-AUC, 13/15 folds).

| Δ vs baseline | Original folds (seed 0) | Fresh folds (seed 100) |
|---|---|---|
| PR-AUC | +0.045 | +0.051 |
| F1 | +0.041 | +0.054 |
| PR-AUC vs `word_1_2` | +0.010 | +0.008 |

The gain on fresh folds is as large as on the folds used for selection, so there is no sign that the Step 2 gain came from choosing among 14 variants on the same folds. Selection bias cannot be ruled out entirely, which is why the test set is still used only once (Step 5).

**2. Mixed groups are mostly a fraud pattern, not label errors.** 8 of the 1,767 multi-posting similarity groups contain both labels (73 postings, 13 fake). In most of them the fake postings copy the description of a real company's posting and remove the company information: all 13 fake postings have an empty `company_profile` and no company logo, while 85% of the real postings in the same groups have both. The labels are therefore consistent with the other fields; the text alone cannot separate these postings. Two groups (389, 2287) join unrelated postings through chains of similar descriptions (a side effect of `connected_components`). The one group where the label looks doubtful (1375, "Data Entry": 3 fake, 1 real with the same missing fields) is in the test set and does not affect development results.

**3. Removing the mixed groups from training does not help.** Without the 66 development postings of mixed groups (10 fake) in the training folds, PR-AUC changes by −0.007 (better in only 5/15 folds) and F1 by +0.002 (9/15). Both differences are well within the fold-to-fold variation (std of the paired difference 0.013 and 0.018). These postings are informative examples of copied-posting fraud rather than noise, so they stay in the training data and the labels are not changed.

**Consequence for Step 3.** The Step 2 configuration (`STEP2_CONFIG`) is confirmed as the new baseline. Step 3 uses the original folds (seed 0), so that its results can be compared directly with the saved Step 1 and Step 2 fold results.